In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
# import shutil
# import os

# drive_path = "/content/drive/MyDrive/data_imagenet/imagenet_resized"
# local_path = "/content/local_data/imagenet_resized"

# # Copy the dataset from Drive to Colab's ultra-fast local disk
# if not os.path.exists(local_path):
#     print("Copying dataset to local Colab disk... (This takes a minute but saves hours)")
#     shutil.copytree(drive_path, local_path)
#     print("Copy complete!")
# else:
#     print("Dataset already exists on local disk.")

In [3]:
!pip install tensorflow

In [4]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.layers import Layer, Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.models import Sequential
import tensorflow as tf
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Concatenate
from tensorflow.keras.models import Model
import tensorflow as tf
import tensorflow as tf
from tensorflow.keras.layers import Dropout
from tensorflow.keras.metrics import Precision, Recall, AUC
from tensorflow.keras import backend as K
import tensorflow as tf
from tensorflow.keras.regularizers import l1
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.utils import to_categorical


In [5]:
# 

In [6]:
class CubixelEdgeDetectionLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(CubixelEdgeDetectionLayer, self).__init__(**kwargs)

    def call(self, inputs):
        adaptive_thresholds = self.calculate_adaptive_threshold(inputs)
        gap_images = tf.map_fn(lambda x: self.extract_features(x[0], x[1]), (inputs, adaptive_thresholds), dtype=tf.float32)
        return gap_images


    @tf.function
    def extract_features(self, image, threshold):
        shape = tf.shape(image)
        height, width = shape[0], shape[1]

        # Vectorized computation of cube sizes
        cube_sizes = self.calculate_cube_sizes(image)  # [height, width, channels]

        # Compute gap differences in a vectorized manner
        gap_differences = self.compute_gap_differences(cube_sizes, threshold, height, width)

        # Edge detection: Identify areas with high gap differences
        edge_image = self.detect_edges(gap_differences)

        return edge_image

    @staticmethod
    def compute_gap_differences(cube_sizes, threshold, height, width):
        # Define shifts for neighbor comparison
        shifts = [(dx, dy) for dx in range(-1, 2) for dy in range(-1, 2) if not (dx == 0 and dy == 0)]

        # Compute gap differences using shifted cube sizes
        gap_differences = tf.zeros([height, width, 3], dtype=tf.float32)  # Updated to have 3 channels
        for dx, dy in shifts:
            shifted_sizes = tf.roll(cube_sizes, shift=[dx, dy], axis=[0, 1])
            gap_differences += tf.abs(cube_sizes - shifted_sizes)

        # Normalize gap differences
        max_gap_difference = tf.reduce_max(gap_differences, axis=[-1], keepdims=True)  # Updated axis for reduction
        normalized_gap_differences = (gap_differences / max_gap_difference) * threshold

        return normalized_gap_differences

    @staticmethod
    def detect_edges(gap_differences):
        # Apply a threshold to identify edges
        edges = tf.cast(gap_differences > 1.0, tf.float32)  # Adjust the threshold as needed

        # Optionally, expand edge dimensions to match the original image's channel count
        edges_expanded = tf.stack([edges, edges, edges], axis=-1)

        return edges_expanded


    @staticmethod
    def calculate_cube_sizes(image):
        return tf.reverse(image, axis=[-1]) / 255  # [b, g, r]
    @staticmethod
    def calculate_adaptive_threshold(pixels):
        cube_sizes = pixels / 255
        avg_size = tf.reduce_mean(cube_sizes, axis=[1, 2, 3])
        size_variability = tf.math.reduce_std(cube_sizes, axis=[1, 2, 3])
        color_diversity = tf.math.reduce_std(pixels, axis=[1, 2, 3])
        color_contrast = tf.map_fn(tf.image.total_variation, pixels)
        return avg_size + size_variability + (color_diversity + color_contrast) / 255

    @staticmethod
    def compute_gaps_vectorized(image, cube_positions, cube_sizes, threshold, height, width):
        # Define shifts for neighbor comparison
        shifts = [(dx, dy) for dx in range(-1, 2) for dy in range(-1, 2) if not (dx == 0 and dy == 0)]

        # Compute gaps using shifted positions and sizes
        gap_image = tf.zeros([height, width, 3], dtype=tf.float32)
        for dx, dy in shifts:
            shifted_positions = tf.roll(cube_positions, shift=[dx, dy], axis=[0, 1])
            shifted_sizes = tf.roll(cube_sizes, shift=[dx, dy], axis=[0, 1])
            gap_image += tf.cast(tf.abs(image - shifted_sizes) > threshold, tf.float32)

        return gap_image
    @staticmethod
    def calculate_cube_positions(height, width):
        y_coords, x_coords = tf.meshgrid(tf.range(width), tf.range(height))
        return tf.stack([tf.cast(x_coords, tf.float32), tf.cast(y_coords, tf.float32), tf.zeros([height, width])], axis=-1)
    def compute_detailed_gaps(self, expanded_image, cube_positions, cube_sizes, threshold, height, width):
        # Define an expanded range of shifts for a larger neighborhood comparison
        extended_shifts = [(dx, dy) for dx in range(-2, 3) for dy in range(-2, 3) if not (dx == 0 and dy == 0)]

        # Initialize gap image
        detailed_gap_image = tf.zeros([height, width, 3], dtype=tf.float32)

        # Compute gaps with extended neighborhood
        for dx, dy in extended_shifts:
            shifted_positions = tf.roll(cube_positions, shift=[dx, dy], axis=[0, 1])
            shifted_sizes = tf.roll(cube_sizes, shift=[dx, dy], axis=[0, 1])
            detailed_gap_image += tf.cast(tf.abs(expanded_image - shifted_sizes) > threshold, tf.float32)

        # Further processing can be added here if needed
        return detailed_gap_image


# # Define the input shape
# input_shape = (32, 32, 3)  # Example input shape, adjust as needed

# # Define the model
# inputs = Input(shape=input_shape)

# # Convolutional and MaxPooling layers
# x = Conv2D(32, kernel_size=(3, 3), activation='relu', padding='same')(inputs)
# x = MaxPooling2D(pool_size=(2, 2))(x)
# x = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(x)
# x = MaxPooling2D(pool_size=(2, 2))(x)
# x = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(x)
# x = MaxPooling2D(pool_size=(2, 2))(x)
# x = Conv2D(256, kernel_size=(3, 3), activation='relu', padding='same')(x)
# x = MaxPooling2D(pool_size=(2, 2))(x)


# # Custom 3D Cube Layer
# cube_layer = CubixelEdgeDetectionLayer()(inputs)
# flattened_cube_output = Flatten()(cube_layer)

# # Concatenate with previous flattened layer
# concatenated = Concatenate()([flattened_cube_output, Flatten()(x)])




# # Output layer
# outputs = Dense(10, activation='softmax')(concatenated)

# # Compile the model
# model = Model(inputs=inputs, outputs=outputs)
# model.compile(optimizer='adam',
#               loss='categorical_crossentropy',
#               metrics=['accuracy',  Precision(name='precision'), Recall(name='recall'), AUC(name='auc')])
# model.summary()
# (x_train, y_train), (x_test, y_test) = cifar10.load_data()
# y_train = to_categorical(y_train, 10)
# y_test = to_categorical(y_test, 10)
# early_stopping = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=10, verbose=1, restore_best_weights=True)

# # Training the model
# history = model.fit(x_train, y_train, epochs=100, validation_data=(x_test, y_test))



# # Using history to print training metrics
# print("\nTraining Metrics:")
# print(f"Loss: {history.history['loss'][-1]}")
# print(f"Accuracy: {history.history['accuracy'][-1]}")
# print(f"Precision: {history.history['precision'][-1]}")
# print(f"Recall: {history.history['recall'][-1]}")
# f1 = 2 * (history.history['precision'][-1] * history.history['recall'][-1]) / (history.history['precision'][-1] + history.history['recall'][-1])
# print(f"F1 Score: {f1}")
# print(f"AUC: {history.history['auc'][-1]}")

# # Using history to print validation metrics
# print("\nValidation Metrics:")
# print(f"Loss: {history.history['val_loss'][-1]}")
# print(f"Accuracy: {history.history['val_accuracy'][-1]}")
# print(f"Precision: {history.history['val_precision'][-1]}")
# print(f"Recall: {history.history['val_recall'][-1]}")
# val_f1 = 2 * (history.history['val_precision'][-1] * history.history['val_recall'][-1]) / (history.history['val_precision'][-1] + history.history['val_recall'][-1])
# print(f"Val F1 Score: {val_f1}")
# print(f"Val AUC: {history.history['val_auc'][-1]}")



In [7]:
#Feature Selection Model 2 (HSVCubixelLayer No RGB)
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
import tensorflow as tf
import numpy as np
from tensorflow.keras.layers import Layer, Conv2D, MaxPooling2D, Flatten, Dense

from tensorflow.keras.models import Sequential
import tensorflow as tf
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Concatenate
from tensorflow.keras.models import Model
import tensorflow as tf
import tensorflow as tf
from tensorflow.keras.layers import Dropout
from tensorflow.keras.metrics import Precision, Recall, AUC
from tensorflow.keras import backend as K
import tensorflow as tf
from tensorflow.keras.regularizers import l1
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
import tensorflow as tf
import tensorflow as tf
import numpy as np
class HSVCubixelLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(HSVCubixelLayer, self).__init__(**kwargs)

    def call(self, inputs):
        # Calculate adaptive thresholds
        adaptive_thresholds = self.calculate_adaptive_threshold(inputs)

        # Extract features for each image and threshold pair
        features = tf.map_fn(lambda x: self.extract_features(x[0], x[1]),
                             (inputs, adaptive_thresholds), dtype=tf.float32)
        return features

    @tf.function
    def extract_features(self, image, threshold):
        shape = tf.shape(image)
        height, width = shape[0], shape[1]

        # Separate processing of RGB and HSV cube sizes
        hsv_cube_sizes = self.calculate_hsv_cube_sizes(image)

        # Calculate properties for RGB and HSV separately
        hsv_properties = self.calculate_cube_properties(hsv_cube_sizes)

        # Compute similarities for RGB and HSV separately
        hsv_similarities = self.compute_similarity(hsv_properties, threshold, height, width)

        # Combine or process the similarities
        edge_image = self.detect_edges(hsv_similarities)

        return edge_image


    @staticmethod
    def calculate_hsv_cube_sizes(image):
        # Convert to HSV and calculate HSV cube sizes
        hsv_cube_sizes = tf.image.rgb_to_hsv(image)
        return hsv_cube_sizes

    @staticmethod
    def calculate_cube_properties(cube_sizes):
        volume = tf.reduce_prod(cube_sizes, axis=-1)
        aspect_ratio = cube_sizes[..., 0] / cube_sizes[..., 1]
        surface_area = cube_sizes[..., 0] * cube_sizes[..., 1]

        # Combine all properties
        combined_properties = tf.stack([volume, aspect_ratio, surface_area], axis=-1)
        return combined_properties

    @staticmethod
    def compute_similarity(cube_properties, threshold, height, width):
        # Initialize with zeros
        similarity = tf.zeros([height, width, 3], dtype=tf.float32)

        # Compute differences in cube properties (placeholder logic)
        shifts = [(dx, dy) for dx in range(-1, 2) for dy in range(-1, 2) if not (dx == 0 and dy == 0)]
        for dx, dy in shifts:
            shifted_properties = tf.roll(cube_properties, shift=[dx, dy], axis=[0, 1])
            similarity += tf.abs(cube_properties - shifted_properties)

        # Normalize similarity
        max_similarity = tf.reduce_max(similarity, axis=[-1], keepdims=True)
        normalized_similarity = (similarity / max_similarity) * threshold
        return normalized_similarity

    @staticmethod
    def detect_edges(similarity):
        edges = tf.cast(similarity > 1.0, tf.float32)  # Threshold for edge detection
        return edges
    @staticmethod
    def calculate_adaptive_threshold(pixels):
        cube_sizes = pixels / 255
        avg_size = tf.reduce_mean(cube_sizes, axis=[1, 2, 3])
        size_variability = tf.math.reduce_std(cube_sizes, axis=[1, 2, 3])
        color_diversity = tf.math.reduce_std(pixels, axis=[1, 2, 3])
        color_contrast = tf.map_fn(tf.image.total_variation, pixels)
        return avg_size + size_variability + (color_diversity + color_contrast) / 255

class AttentionLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)

    def call(self, inputs):
        cnn_features, similarity_features = inputs

        # Resize similarity features to match cnn_features spatial dimensions
        similarity_features_resized = tf.image.resize(
            similarity_features, (tf.shape(cnn_features)[1], tf.shape(cnn_features)[2])
        )

        # Expand dimensions to match cnn_features
        similarity_features_expanded = tf.expand_dims(similarity_features_resized, -1)

        # Reduce the expanded similarity features to the channel size of cnn_features
        similarity_features_reduced = tf.reduce_mean(similarity_features_expanded, axis=-2)

        # Apply attention
        attended_features = cnn_features * similarity_features_reduced

        return attended_features





# # Define the input shape and the model
# input_shape = (32, 32, 3)
# inputs = Input(shape=input_shape)

# # Convolutional Stream (Original Image Data)
# x = Conv2D(32, kernel_size=(3, 3), activation='relu', padding='same')(inputs)
# x = MaxPooling2D(pool_size=(2, 2))(x)
# x = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(x)
# x = MaxPooling2D(pool_size=(2, 2))(x)

# # Custom3DCubeLayer Output (Similarity Matrix)
# cube_layer_output = HSVCubixelLayer()(inputs)
# cube_layer_output = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(cube_layer_output)
# cube_layer_output = MaxPooling2D(pool_size=(2, 2))(cube_layer_output)
# # Additional downsampling to match 'x' dimensions
# cube_layer_output = MaxPooling2D(pool_size=(2, 2))(cube_layer_output)

# # Merge the Convolutional Stream with Cube Layer Output
# merged = Concatenate()([x, cube_layer_output])

# # Continue with further Convolutional Layers
# y = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(merged)
# y = MaxPooling2D(pool_size=(2, 2))(y)

# # Flatten and add Dense Layers
# y = Flatten()(y)
# y = Dense(128, activation='relu')(y)

# # Output layer
# outputs = Dense(10, activation='softmax')(y)

# # Create the model
# model = Model(inputs=inputs, outputs=outputs)

# # Create the model

# model.compile(optimizer='adam',
#               loss='categorical_crossentropy',
#               metrics=['accuracy',  Precision(name='precision'), Recall(name='recall'), AUC(name='auc')])
# model.summary()
# (x_train, y_train), (x_test, y_test) = cifar10.load_data()
# y_train = to_categorical(y_train, 10)
# y_test = to_categorical(y_test, 10)
# early_stopping = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=10, verbose=1, restore_best_weights=True)

# Training the model
# history = model.fit(x_train, y_train, epochs=100, validation_data=(x_test, y_test))



# # Using history to print training metrics
# print("\nTraining Metrics:")
# print(f"Loss: {history.history['loss'][-1]}")
# print(f"Accuracy: {history.history['accuracy'][-1]}")
# print(f"Precision: {history.history['precision'][-1]}")
# print(f"Recall: {history.history['recall'][-1]}")
# f1 = 2 * (history.history['precision'][-1] * history.history['recall'][-1]) / (history.history['precision'][-1] + history.history['recall'][-1])
# print(f"F1 Score: {f1}")
# print(f"AUC: {history.history['auc'][-1]}")

# # Using history to print validation metrics
# print("\nValidation Metrics:")
# print(f"Loss: {history.history['val_loss'][-1]}")
# print(f"Accuracy: {history.history['val_accuracy'][-1]}")
# print(f"Precision: {history.history['val_precision'][-1]}")
# print(f"Recall: {history.history['val_recall'][-1]}")
# val_f1 = 2 * (history.history['val_precision'][-1] * history.history['val_recall'][-1]) / (history.history['val_precision'][-1] + history.history['val_recall'][-1])
# print(f"Val F1 Score: {val_f1}")
# print(f"Val AUC: {history.history['val_auc'][-1]}")


# # Normalize the data
# x_train_normalized = x_train.astype("float32") / 255

# # Initialize the custom layer
# custom_layer = Custom3DCubeLayer()
# # Apply the custom layer to the first 10 images
# processed_images = custom_layer(x_train_normalized[:10])
# rgb_similarities = processed_images[..., 3:]

# # Set up the plotting
# plt.figure(figsize=(20, 4))
# for i in range(10):
#     # Original image
#     plt.subplot(2, 10, i + 1)
#     plt.imshow(x_train[i])
#     plt.title("Original")
#     plt.axis("off")

#     # Processed image (RGB similarities)
#     plt.subplot(2, 10, i + 11)
#     plt.imshow(rgb_similarities[i])
#     plt.title("RGB Similarities")
#     plt.axis("off")

# # Display the plot
# plt.show()

In [8]:
#Feature Selection Model 2 (RGBHSVCubixelLayer)
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
import tensorflow as tf
import numpy as np
from tensorflow.keras.layers import Layer, Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.models import Sequential
import tensorflow as tf
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Concatenate
from tensorflow.keras.models import Model
import tensorflow as tf
import tensorflow as tf
from tensorflow.keras.layers import Dropout
from tensorflow.keras.metrics import Precision, Recall, AUC
from tensorflow.keras import backend as K
import tensorflow as tf
from tensorflow.keras.regularizers import l1
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
import tensorflow as tf
import tensorflow as tf
import numpy as np
class RGBHSVCubixelLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(RGBHSVCubixelLayer, self).__init__(**kwargs)

    def call(self, inputs):
        # Calculate adaptive thresholds
        adaptive_thresholds = self.calculate_adaptive_threshold(inputs)

        # Extract features for each image and threshold pair
        features = tf.map_fn(lambda x: self.extract_features(x[0], x[1]),
                             (inputs, adaptive_thresholds), dtype=tf.float32)
        return features

    @tf.function
    def extract_features(self, image, threshold):
        shape = tf.shape(image)
        height, width = shape[0], shape[1]


        # Separate processing of RGB and HSV cube sizes
        rgb_cube_sizes = self.calculate_rgb_cube_sizes(image)
        hsv_cube_sizes = self.calculate_hsv_cube_sizes(image)

        # Calculate properties for RGB and HSV separately
        rgb_properties = self.calculate_cube_properties(rgb_cube_sizes)
        hsv_properties = self.calculate_cube_properties(hsv_cube_sizes)

        # Compute similarities for RGB and HSV separately
        rgb_similarities = self.compute_similarity(rgb_properties, threshold, height, width)
        hsv_similarities = self.compute_similarity(hsv_properties, threshold, height, width)

        # Combine or process the similarities
        combined_similarities = tf.concat([rgb_similarities, hsv_similarities], axis=-1)
        edge_image = self.detect_edges(combined_similarities)

        return edge_image

    @staticmethod
    def calculate_rgb_cube_sizes(image):
        # Calculate RGB cube sizes only
        rgb_cube_sizes = image / 255  # Normalizing the RGB values
        return rgb_cube_sizes

    @staticmethod
    def calculate_hsv_cube_sizes(image):
        # Convert to HSV and calculate HSV cube sizes
        hsv_cube_sizes = tf.image.rgb_to_hsv(image)
        return hsv_cube_sizes

    @staticmethod
    def calculate_cube_properties(cube_sizes):
        # Placeholder method to calculate cube properties like volume, aspect ratio, etc.
        # Replace with actual property calculations
        volume = tf.reduce_prod(cube_sizes, axis=-1)  # Simplified volume calculation
        aspect_ratio = cube_sizes[..., 0] / cube_sizes[..., 1]  # Simplified aspect ratio
        surface_area = cube_sizes[..., 0] * cube_sizes[..., 1]  # Simplified surface area

        # Combine all properties
        combined_properties = tf.stack([volume, aspect_ratio, surface_area], axis=-1)
        return combined_properties

    @staticmethod
    def compute_similarity(cube_properties, threshold, height, width):
        # Placeholder method for similarity computation
        # Initialize with zeros
        similarity = tf.zeros([height, width, 3], dtype=tf.float32)

        # Compute differences in cube properties (placeholder logic)
        shifts = [(dx, dy) for dx in range(-1, 2) for dy in range(-1, 2) if not (dx == 0 and dy == 0)]
        for dx, dy in shifts:
            shifted_properties = tf.roll(cube_properties, shift=[dx, dy], axis=[0, 1])
            similarity += tf.abs(cube_properties - shifted_properties)

        # Normalize similarity
        max_similarity = tf.reduce_max(similarity, axis=[-1], keepdims=True)
        normalized_similarity = (similarity / max_similarity) * threshold
        return normalized_similarity

    @staticmethod
    def detect_edges(similarity):
        # Placeholder for edge detection or other feature extraction
        edges = tf.cast(similarity > 1.0, tf.float32)  # Threshold for edge detection
        return edges
    @staticmethod
    def calculate_adaptive_threshold(pixels):
        # Placeholder for adaptive threshold calculation
        cube_sizes = pixels / 255
        avg_size = tf.reduce_mean(cube_sizes, axis=[1, 2, 3])
        size_variability = tf.math.reduce_std(cube_sizes, axis=[1, 2, 3])
        color_diversity = tf.math.reduce_std(pixels, axis=[1, 2, 3])
        color_contrast = tf.map_fn(tf.image.total_variation, pixels)
        return avg_size + size_variability + (color_diversity + color_contrast) / 255


In [9]:
#Feature Selection Model 5. combined_cube_sizes = tf.concat([rgb_cube_sizes, hsv_cube_sizes], axis=-1)
import tensorflow as tf
import numpy as np
from tensorflow.keras.layers import Layer, Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.models import Sequential
import tensorflow as tf
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Concatenate
from tensorflow.keras.models import Model
import tensorflow as tf
import tensorflow as tf
from tensorflow.keras.layers import Dropout
from tensorflow.keras.metrics import Precision, Recall, AUC
from tensorflow.keras import backend as K
import tensorflow as tf
from tensorflow.keras.regularizers import l1
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
import tensorflow as tf

class Custom3DCubeLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(Custom3DCubeLayer, self).__init__(**kwargs)

    def call(self, inputs):
        # Calculate adaptive thresholds
        adaptive_thresholds = self.calculate_adaptive_threshold(inputs)

        # Extract features for each image and threshold pair
        gap_images = tf.map_fn(lambda x: self.extract_features(x[0], x[1]),
                               (inputs, adaptive_thresholds), dtype=tf.float32)
        return gap_images

    @tf.function
    def extract_features(self, image, threshold):
        shape = tf.shape(image)
        height, width = shape[0], shape[1]

        # Calculate cube sizes and properties
        cube_sizes = self.calculate_cube_sizes(image)
        cube_properties = self.calculate_cube_properties(cube_sizes)

        # Compute similarities based on cube properties
        similarities = self.compute_similarity(cube_properties, threshold, height, width)
        normalized_similarity = tf.math.sigmoid(similarities)  # Example normalization

        # Edge detection or other feature extraction can be done here
        edge_image = self.detect_edges(similarities,threshold)

        return edge_image

    @staticmethod
    def calculate_cube_sizes(image):
        # Calculate RGB cube sizes (placeholder)
        rgb_cube_sizes = tf.reverse(image, axis=[-1]) / 255  # [b, g, r]

        # Convert to HSV and calculate HSV cube sizes (placeholder)
        hsv_image = tf.image.rgb_to_hsv(image)
        hsv_cube_sizes = hsv_image

        # Combine both RGB and HSV cube sizes
        combined_cube_sizes = tf.concat([rgb_cube_sizes, hsv_cube_sizes], axis=-1)
        return combined_cube_sizes

    @staticmethod
    def calculate_cube_properties(cube_sizes):

        volume = tf.reduce_prod(cube_sizes, axis=-1)  # Simplified volume calculation
        aspect_ratio = cube_sizes[..., 0] / cube_sizes[..., 1]  # Simplified aspect ratio
        surface_area = cube_sizes[..., 0] * cube_sizes[..., 1]  # Simplified surface area

        # Combine all properties
        combined_properties = tf.stack([volume, aspect_ratio, surface_area], axis=-1)
        return combined_properties

    @staticmethod
    def compute_similarity(cube_properties, threshold, height, width):

        similarity = tf.zeros([height, width, 3], dtype=tf.float32)

        # Compute differences in cube properties (placeholder logic)
        shifts = [(dx, dy) for dx in range(-1, 2) for dy in range(-1, 2) if not (dx == 0 and dy == 0)]
        for dx, dy in shifts:
            shifted_properties = tf.roll(cube_properties, shift=[dx, dy], axis=[0, 1])
            similarity += tf.abs(cube_properties - shifted_properties)

        # Normalize similarity
        max_similarity = tf.reduce_max(similarity, axis=[-1], keepdims=True)
        normalized_similarity = (similarity / max_similarity) * threshold
        return normalized_similarity


    def detect_edges(self, similarity, threshold):
        # Calculate the total difference in similarity
        total_diff = similarity  # Adjust this based on your actual calculation

        # Apply different levels of significance based on the threshold
        levels = tf.where(
            total_diff > threshold * 2, 3,  # Maximum gap
            tf.where(
                total_diff > threshold, 2,  # Moderate gap
                tf.where(
                    total_diff > threshold / 2, 1,  # Minor gap
                    0  # No significant gap
                )
            )
        )

        return tf.cast(levels, tf.float32)

    @staticmethod
    def calculate_adaptive_threshold(pixels):
        cube_sizes = pixels / 255
        avg_size = tf.reduce_mean(cube_sizes, axis=[1, 2, 3])
        size_variability = tf.math.reduce_std(cube_sizes, axis=[1, 2, 3])
        color_diversity = tf.math.reduce_std(pixels, axis=[1, 2, 3])
        color_contrast = tf.map_fn(tf.image.total_variation, pixels)
        return avg_size + size_variability + (color_diversity + color_contrast) / 255

# Define the input shape
# input_shape = (32, 32, 3)  # Example input shape, adjust as needed

# # Define the model
# inputs = Input(shape=input_shape)
# # Convolutional and MaxPooling layers
# x = Conv2D(32, kernel_size=(3, 3), activation='relu', padding='same')(inputs)
# x = MaxPooling2D(pool_size=(2, 2))(x)
# x = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(x)
# x = MaxPooling2D(pool_size=(2, 2))(x)
# x = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(x)
# x = MaxPooling2D(pool_size=(2, 2))(x)
# x = Conv2D(256, kernel_size=(3, 3), activation='relu', padding='same')(x)
# x = MaxPooling2D(pool_size=(2, 2))(x)

# # Custom 3D Cube Layer
# cube_layer = Custom3DCubeLayer()(inputs)

# flattened_cube_output = Flatten()(cube_layer)

# # Concatenate with previous flattened layer
# concatenated = Concatenate()([flattened_cube_output, Flatten()(x)])




# # Output layer
# outputs = Dense(10, activation='softmax')(concatenated)

# # Compile the model
# model = Model(inputs=inputs, outputs=outputs)
# model.compile(optimizer='adam',
#               loss='categorical_crossentropy',
#               metrics=['accuracy',  Precision(name='precision'), Recall(name='recall'), AUC(name='auc')])
# (x_train, y_train), (x_test, y_test) = cifar10.load_data()
# y_train = to_categorical(y_train, 10)
# y_test = to_categorical(y_test, 10)
# early_stopping = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=10, verbose=1, restore_best_weights=True)

# # Training the model
# history = model.fit(x_train, y_train, epochs=100, validation_data=(x_test, y_test))



# # Using history to print training metrics
# print("\nTraining Metrics:")
# print(f"Loss: {history.history['loss'][-1]}")
# print(f"Accuracy: {history.history['accuracy'][-1]}")
# print(f"Precision: {history.history['precision'][-1]}")
# print(f"Recall: {history.history['recall'][-1]}")
# f1 = 2 * (history.history['precision'][-1] * history.history['recall'][-1]) / (history.history['precision'][-1] + history.history['recall'][-1])
# print(f"F1 Score: {f1}")
# print(f"AUC: {history.history['auc'][-1]}")

# # Using history to print validation metrics
# print("\nValidation Metrics:")
# print(f"Loss: {history.history['val_loss'][-1]}")
# print(f"Accuracy: {history.history['val_accuracy'][-1]}")
# print(f"Precision: {history.history['val_precision'][-1]}")
# print(f"Recall: {history.history['val_recall'][-1]}")
# val_f1 = 2 * (history.history['val_precision'][-1] * history.history['val_recall'][-1]) / (history.history['val_precision'][-1] + history.history['val_recall'][-1])
# print(f"Val F1 Score: {val_f1}")
# print(f"Val AUC: {history.history['val_auc'][-1]}")



In [10]:
# !nvidia-smi

In [11]:
#Cubixel Kernel
class CubixelLayer(tf.keras.layers.Layer):
    def __init__(self, num_kernels, kernel_size=(3, 3), volume_threshold=1.0, **kwargs):
        super(CubixelLayer, self).__init__(**kwargs)
        self.num_kernels = num_kernels
        self.kernel_size = kernel_size
        self.volume_threshold = volume_threshold
        self.edge_color = np.array([255, 0, 0], dtype=np.float32)  # Red color for edges
        self.kernels = self.add_weight(
            name='kernels',
            shape=(num_kernels, kernel_size[0], kernel_size[1], 1, 1),
            initializer='random_normal',
            trainable=True
        )

    def call(self, inputs):
        # Calculate cubixel properties
        volumes, aspect_ratios = self.calculate_cubixel_properties(inputs)

        kernel_outputs = []
        for i in range(self.num_kernels):
            kernel = self.kernels[i]
            altered_volumes, altered_aspect_ratios = self.apply_magnetic_kernel((volumes, aspect_ratios), kernel)
            local_threshold = self.calculate_local_threshold(altered_volumes)

            # Detect edges at the current scale
            edges = tf.cast(altered_volumes, tf.float32) > tf.cast(local_threshold, tf.float32)
            edges |= tf.cast(altered_aspect_ratios, tf.float32) > tf.cast(local_threshold, tf.float32)

            # Apply edge color and store each kernel's output separately
            edge_colored_image = tf.cast(edges, tf.float32)[..., tf.newaxis] * self.edge_color
            kernel_outputs.append(edge_colored_image)

        # Stack all kernel outputs along a new axis (axis=0)
        # This will give you a tensor of shape [num_kernels, height, width, channels]
        multi_scale_edges = tf.stack(kernel_outputs, axis=0)

        return multi_scale_edges


    @staticmethod
    def calculate_cubixel_properties(image):
        # Normalize the image to a range of 0-1
        image_normalized = image / 255.0

        # Calculate volumes as the product of the RGB values (considered as dimensions)
        # Assuming image shape: [batch, height, width, channels]
        # Volumes are calculated for each pixel
        volumes = image_normalized[..., 0] * image_normalized[..., 1] * image_normalized[..., 2]

        # Calculate aspect ratios
        # Example: You could define aspect ratios based on the relative proportions of RGB values
        # Adjust the calculation based on your specific definition of aspect ratio
        aspect_ratios = image_normalized[..., 0] / image_normalized[..., 1]  # Placeholder, replace with your aspect ratio calculation

        return volumes, aspect_ratios

    def apply_magnetic_kernel(self, cubixel_properties, kernel):
        kernel_tf = tf.reshape(kernel, [self.kernel_size[0], self.kernel_size[1], 1, 1])
        # Ensure that the input to the convolution is 4D by adding a channel dimension.
        altered_volume = tf.nn.conv2d(cubixel_properties[0][..., tf.newaxis], kernel_tf, strides=[1, 1, 1, 1], padding="SAME")
        altered_aspect_ratio = tf.nn.conv2d(cubixel_properties[1][..., tf.newaxis], kernel_tf, strides=[1, 1, 1, 1], padding="SAME")
        # Remove the last dimension to maintain the original shape.
        return altered_volume[..., 0], altered_aspect_ratio[..., 0]
    def calculate_local_threshold(self, image, neighborhood_size=9):
        pad_size = neighborhood_size // 2
        image = tf.expand_dims(image, -1) if len(image.shape) == 3 else image

        padded_images = tf.pad(
            image,
            paddings=[[0, 0], [pad_size, pad_size], [pad_size, pad_size], [0, 0]],
            mode='REFLECT'
        )

        patches = tf.image.extract_patches(
            images=padded_images,
            sizes=[1, neighborhood_size, neighborhood_size, 1],
            strides=[1, 1, 1, 1],
            rates=[1, 1, 1, 1],
            padding='VALID'
        )

        local_thresholds = tf.reduce_mean(patches, axis=-1, keepdims=True)

        # Ensure the output has the same shape as the input, minus the channel dimension
        local_thresholds = tf.squeeze(local_thresholds, -1)

        return local_thresholds


In [ ]:
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Concatenate, Dropout, BatchNormalization, GlobalAveragePooling2D, Lambda
from tensorflow.keras.models import Model
import tensorflow as tf


class CombinedCubixelModel:
    def __init__(self, input_shape=(32,32,3), num_classes=10):
        self.input_shape = input_shape
        self.num_classes = num_classes


    def build(self):
       

        raw_inputs = Input(shape=self.input_shape)


            # 1. Data Augmentation
        aug_inputs = tf.keras.Sequential([
                tf.keras.layers.RandomFlip("horizontal"),
                tf.keras.layers.RandomRotation(0.1),
                tf.keras.layers.RandomZoom(0.1),
            ])(raw_inputs)

            #cubixel layers
        edge_map = CubixelEdgeDetectionLayer()(aug_inputs)
        edge_map_4d = tf.keras.layers.Reshape((64, 64, 9))(edge_map)

        hsv_map = HSVCubixelLayer()(aug_inputs)
        rgbhsv_map = RGBHSVCubixelLayer()(aug_inputs)
        custom_3d = Custom3DCubeLayer()(aug_inputs)


        cub_layer_5D = CubixelLayer(num_kernels=6, kernel_size=(3, 3), volume_threshold=1, trainable=False)(aug_inputs)
                    #using tf.stack puts kernel first in this layer but keras expects batch size to be first
        cub_transposed = Lambda(lambda x: tf.transpose(x, perm=[1, 0, 2, 3, 4]))(cub_layer_5D)
        #combining kernel*channels so dimensions match
        cub_layer_4D = Lambda(lambda x: tf.reshape(x, (-1, tf.shape(x)[2], tf.shape(x)[3], tf.shape(x)[1] * tf.shape(x)[4])))(cub_transposed)            
        cub_norm = BatchNormalization()(cub_layer_4D)


        merged_spatial = Concatenate()([
                edge_map_4d,
                hsv_map,
                rgbhsv_map,
                custom_3d,
                cub_norm
            ])
        x = Conv2D(16, (1, 1), activation='relu')(merged_spatial)
        x=BatchNormalization()(x)
        x = Flatten()(x)

            # dense layer
        x = Dense(32, activation='relu')(x)
        x=Dropout(0.5)(x)
        x = Dense(128, activation='relu', name="high_level_embedding")(x) # Your embedding
        x = BatchNormalization()(x)
        outputs = Dense(self.num_classes, activation="softmax")(x)

        model = Model(inputs=raw_inputs, outputs=outputs, name="CombinedCubixelModel")
        return model

In [24]:
model_builder=CombinedCubixelModel(input_shape=(64,64,3),num_classes=1000)
model=model_builder.build()
model.compile(optimizer='adam',
              jit_compile=True,
              loss='categorical_crossentropy',
              metrics=['accuracy',  Precision(name='precision'), Recall(name='recall'), AUC(name='auc')])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/trainer.py:212: UserWarning: Model doesn't support `jit_compile=True`. Proceeding with `jit_compile=False`.
  warnings.warn(


Model: "CombinedCubixelModel"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 64, 64, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_1        │ (None, 64, 64, 3) │          0 │ input_layer_2[0]… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cubixel_layer_1     │ (6, None, 64, 64, │         54 │ sequential_1[0][… │
│ (CubixelLayer)      │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_2 (Lambda)   │ (None, 6, 64, 64, │          0 │ cubixel_layer_1[… │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cubixel_edge_detec… │ (None, 64, 64, 3, │          0 │ sequential_1[0][… │
│ (CubixelEdgeDetect… │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_3 (Lambda)   │ (None, 64, 64,    │          0 │ lambda_2[0][0]    │
│                     │ 18)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 64, 64, 9) │          0 │ cubixel_edge_det… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hsv_cubixel_layer_1 │ (None, 64, 64, 3) │          0 │ sequential_1[0][… │
│ (HSVCubixelLayer)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rgbhsv_cubixel_lay… │ (None, 64, 64, 6) │          0 │ sequential_1[0][… │
│ (RGBHSVCubixelLaye… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ custom3d_cube_laye… │ (None, 64, 64, 3) │          0 │ sequential_1[0][… │
│ (Custom3DCubeLayer) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │         72 │ lambda_3[0][0]    │
│ (BatchNormalizatio… │ 18)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 64, 64,    │          0 │ reshape_1[0][0],  │
│ (Concatenate)       │ 39)               │            │ hsv_cubixel_laye… │
│                     │                   │            │ rgbhsv_cubixel_l… │
│                     │                   │            │ custom3d_cube_la… │
│                     │                   │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 64, 64,    │        640 │ concatenate_1[0]… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │         64 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 65536)     │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 16)        │  1,048,592 │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 16)        │          0 │ dense_2[0][0]   

 Total params: 1,181,110 (4.51 MB)

 Trainable params: 1,180,732 (4.50 MB)

 Non-trainable params: 378 (1.48 KB)

In [14]:
# (x_train, y_train), (x_test, y_test) = cifar10.load_data()
# x_train = x_train.astype('float32') / 255.0
# x_test = x_test.astype('float32') / 255.0

# y_train = to_categorical(y_train, 10)
# y_test = to_categorical(y_test, 10)
# early_stopping = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=10, verbose=1, restore_best_weights=True)

# #Training the model
# history = model.fit(x_train, y_train, epochs=30, validation_data=(x_test, y_test))




# # Using history to print training metrics
# print("\nTraining Metrics:")
# print(f"Loss: {history.history['loss'][-1]}")
# print(f"Accuracy: {history.history['accuracy'][-1]}")
# print(f"Precision: {history.history['precision'][-1]}")
# print(f"Recall: {history.history['recall'][-1]}")
# f1 = 2 * (history.history['precision'][-1] * history.history['recall'][-1]) / (history.history['precision'][-1] + history.history['recall'][-1])
# print(f"F1 Score: {f1}")
# print(f"AUC: {history.history['auc'][-1]}")

# # Using history to print validation metrics
# print("\nValidation Metrics:")
# print(f"Loss: {history.history['val_loss'][-1]}")
# print(f"Accuracy: {history.history['val_accuracy'][-1]}")
# print(f"Precision: {history.history['val_precision'][-1]}")
# print(f"Recall: {history.history['val_recall'][-1]}")
# val_f1 = 2 * (history.history['val_precision'][-1] * history.history['val_recall'][-1]) / (history.history['val_precision'][-1] + history.history['val_recall'][-1])
# print(f"Val F1 Score: {val_f1}")
# print(f"Val AUC: {history.history['val_auc'][-1]}")


In [15]:
# evaluation
# for model with GAP and Batch Normalization without augmentation
# model shows high overfitting with accuracy being 0.7404 and val accuracy being 0.5450. Val precision is 0.6389 and train is 0.8536
# max accuracy for train is 0.77 after 30 epochs and continued to steadily rise whereas val accuracy was 0.5437

# next trying normalization and a deeper dense layer
# got a max train accuracy of 0.7005 and val of 0.5963 (went down after that and started overfitting)
# val precision maximum in any epoch 73%

# cannot get accuracy higher than 61%

In [25]:
import tensorflow_datasets as tfds

(ds_train, ds_test), ds_info = tfds.load(
    'imagenet_resized/64x64',
    data_dir="/content/drive/MyDrive/data_imagenet", 
    split=["train", "validation"],
    shuffle_files=True,
    as_supervised=True,
    with_info=True,
    download=True 
)

In [37]:
def preprocess(image,label):
  image=tf.cast(image,tf.float32)/255.0
  label=tf.one_hot(label,1000)
  return image,label



# In your data pipeline
train_ds = ds_train.map(preprocess).shuffle(1000).batch(64).prefetch(tf.data.AUTOTUNE)


# In your data pipeline
test_ds = ds_test.map(preprocess).cache().batch(64).prefetch(tf.data.AUTOTUNE)




In [ ]:
from datetime import datetime
import pandas as pd
import os

def convert_history_csv(history,time_taken,csv_name="training_logs.csv",model_name="CombinedCubixelModel"):
  new_data = {metric: values[-1] for metric, values in history.history.items()}
  new_data["model"]=model_name
  new_data["date"]=datetime.now()
  new_data['epochs_run'] = len(history.history['loss'])
  new_data["time_taken"]=time_taken

  df_new=pd.DataFrame([new_data])

    # 3. Handle File Writing
  if not os.path.isfile(csv_name):
    df_new.to_csv(csv_name, index=False)
    print(f"Created new log file: {csv_name}")
  else:
    # Append to existing file without rewriting the header
    df_new.to_csv(csv_name, mode='a', index=False, header=False)
    print(f"Appended metrics to: {csv_name}")


In [ ]:
import time 
from tensorflow.keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(
    monitor='val_loss',       
    patience=3,              
    restore_best_weights=True  
)

start_time=time.perf_counter()

history = model.fit(
    train_ds,
    epochs=1,
    validation_data=test_ds,
    callbacks=[early_stopping]
)

end_time=time.perf_counter()
time_taken=end_time-start_time
convert_history_csv(history,time_taken)

 1554/20019 ━━━━━━━━━━━━━━━━━━━━ 1:48:32 353ms/step - accuracy: 0.0033 - auc: 0.5055 - loss: 6.8094 - precision: 0.0000e+00 - recall: 0.0000e+00

KeyboardInterrupt: 